# 2.3.1 — Word count distribuito sul full text di CORD-19

Questo notebook **importa** `word_count.py`: non riscrive l'algoritmo e non lancia
sottoprocessi. Tutta la logica sta nel modulo, qui c'è l'esecuzione e la lettura dei
risultati, così esiste una sola sorgente di verità.

L'algoritmo è quello dell'assignment (§2.3.1):

| fase | cosa produce |
|---|---|
| **Map** | per ogni documento *D*, le coppie `(w, cp(w))` — quante volte la parola `w` compare in *D* |
| **Reduce** | per ogni parola `w`, `c(w) = Σ cp(w)` su tutti i documenti |

Struttura dati: **Bag**, come raccomanda il testo («we recommend utilizing the RDD/Bag
data structure»).

Input: `data/silver/paragraphs` — una riga per paragrafo, già sanificato dalla pipeline
di conversione (vedi `DATA_DICTIONARY.md`).

Le scelte di pulizia del testo sono tutte motivate da misure sul corpus: la
giustificazione riga per riga è nel README di questa cartella.

## 1 · Cluster

Dove gira il calcolo lo decide `cluster.txt` alla root del repo (git-ignored), non il
codice: sul Mac parte un `LocalCluster`, sulla VM un `SSHCluster` sui nodi elencati.
Lo stesso notebook gira nei due posti senza modifiche.

In [ ]:
import sys
import time
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "Giulia" else Path.cwd()
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "Giulia"))

from bench import get_client, sweep, unmanaged_memory
import word_count as wc

client, cluster = get_client(repo_root=REPO)
client

## 2 · I dati

`read_paragraphs` legge tre colonne e scarta i paragrafi `is_reference_like`
(dichiarazioni di conflitto d'interesse, author contributions, funding: l'1,24% del
corpus). Il filtro sta sul DataFrame, prima di passare al Bag, perché una maschera
vettoriale su una colonna booleana costa molto meno che testare ogni elemento.

In [ ]:
INPUT = REPO / "data" / "silver" / "paragraphs"

paragraphs = wc.read_paragraphs(INPUT)
print("partitions:", paragraphs.npartitions)
paragraphs.take(1)

## 3 · Come si comporta la sanitizzazione

Un controllo a occhio prima di lanciare il calcolo vero: cosa resta di un testo con
trattini tipografici, lettere greche e stop-word.

In [ ]:
demo = "The SARS–CoV-2 virus and TNF-α were measured at 5 µg/mL in Müller's study."
print(wc.sanitize(demo))
print(wc.words(demo))

## 4 · Il grafo delle due fasi

`word_count` restituisce i due Bag, ancora **lazy**: nulla è stato calcolato.

Da guardare: `doc_counts` ha lo stesso numero di partizioni dell'input — la fase Map è
puramente locale, il conteggio per documento avviene *dentro* la partizione (è il
*combiner* del MapReduce classico). `global_counts` ne ha **una**, perché `foldby`
riduce a una partizione sola: è sostenibile solo perché la chiave è la parola, il cui
spazio satura al crescere del corpus.

In [ ]:
doc_counts, global_counts = wc.word_count(paragraphs)
print("doc_counts   partitions:", doc_counts.npartitions)
print("global_counts partitions:", global_counts.npartitions)

## 5 · Smoke test

Prima del run completo, le stesse identiche operazioni su poche partizioni.

In [ ]:
smoke = wc.read_paragraphs(INPUT, npartitions=8)
_, smoke_counts = wc.word_count(smoke)
smoke_counts.topk(10, key=1).compute()

## 6 · Run completo

`topk` pota presto: ogni partizione inoltra solo le sue prime N, quindi non serve
materializzare tutto il vocabolario per avere la classifica.

In [ ]:
TOP_N = 20

started = time.perf_counter()
top = global_counts.topk(TOP_N, key=1).compute()
elapsed = time.perf_counter() - started
print(f"{elapsed:.1f} s")

for word, count in top:
    print(f"{count:>12,}  {word}")

## 7 · Verifica dell'invariante

Il Reduce **raggruppa e basta**: non perde né inventa occorrenze. Il controllo somma i
conteggi prima e dopo e verifica che coincidano.

Nessun totale assoluto scritto a mano: il dump locale e il corpus della VM sono dataset
diversi, quindi l'unica garanzia sensata è strutturale (`PROJECT_CONTEXT.md`, regola 8.2).

Costa: sommare i conteggi per-documento obbliga a percorrere tutta la tabella
intermedia, mentre `topk` può potare. Si lancia in sviluppo e prima di una consegna,
non a ogni esecuzione.

In [ ]:
import dask

after_map, after_reduce = dask.compute(doc_counts.pluck(1).sum(), global_counts.pluck(1).sum())
assert after_map == after_reduce, (after_map, after_reduce)
print(f"invariante ok: {after_map:,} occorrenze prima e dopo il reduce")

## 8 · Barplot

Il grafico che l'assignment chiede esplicitamente («create a barplot of the top
words»).

In [ ]:
OUT = REPO / "reports" / "word_count"
OUT.mkdir(parents=True, exist_ok=True)

wc.barplot(top, OUT / "top_words.png", f"Top {len(top)} words in the CORD-19 body text")

from IPython.display import Image
Image(str(OUT / "top_words.png"))

## 9 · Benchmark obbligatori

Le linee guida del corso li richiedono esplicitamente: tempo di esecuzione contro
**numero di partizioni** e contro **numero di worker**. Senza, il progetto è considerato
incompleto.

Due precisazioni che cambiano il significato della misura:

- nel sweep sulle partizioni **i dati restano gli stessi** e cambia solo come sono
  suddivisi. Misurare su fette di corpus via via più grandi misurerebbe la dimensione
  del dato, che è un'altra domanda;
- si usa `repartition(npartitions=k)`, **non** `repartition(partition_size=...)`: la
  seconda si impianta sotto un Client distribuito, perché per stimare le taglie deve
  materializzare (`PROJECT_CONTEXT.md`, §8.4);
- su `SSHCluster` il pool di worker è la lista di host, quindi il sweep può solo
  **scendere**: si mettono in `cluster.txt` tutti i worker e si scala verso il basso.

`bench.measure` ripete ogni misura più volte, chiama `sweep()` **tra** una ripetizione e
l'altra (mai dentro una regione cronometrata) e non usa mai `client.restart()`, che
sporcherebbe i tempi. Tiene tutte le ripetizioni, non solo la media: è la dispersione a
dire se una differenza tra due configurazioni è reale.

In [ ]:
BENCH_PARTITIONS = 400   # fetta di corpus su cui si misura, FISSA per tutto il benchmark
BENCH_REPEATS = 3

base = wc.read_paragraphs(INPUT, npartitions=BENCH_PARTITIONS)
print("partizioni native della fetta:", base.npartitions)
records = []

### 9.1 · Tempo vs numero di partizioni

In [ ]:
from bench import measure, scale, save, plot_scaling

for k in (50, 100, 200, 400):
    _, counts = wc.word_count(base.repartition(npartitions=k))
    records += measure(
        lambda: counts.topk(TOP_N, key=1).compute(),
        client, repeats=BENCH_REPEATS, sweep="partitions", partitions=k,
    )

### 9.2 · Tempo vs numero di worker

Partizionamento fisso, cambia solo quanta macchina lavora.

In [ ]:
_, counts = wc.word_count(base)

for n_workers in (3, 2, 1):          # a scendere: vedi la nota su SSHCluster
    scale(client, cluster, n_workers)
    records += measure(
        lambda: counts.topk(TOP_N, key=1).compute(),
        client, repeats=BENCH_REPEATS, sweep="workers", partitions=base.npartitions,
    )

### 9.3 · Risultati

In [ ]:
save(records, OUT / "benchmark.csv")

by_partitions = [r for r in records if r["sweep"] == "partitions"]
by_workers = [r for r in records if r["sweep"] == "workers"]
plot_scaling(by_partitions, "partitions", OUT / "bench_partitions.png", "Word count: time vs partitions")
plot_scaling(by_workers, "workers", OUT / "bench_workers.png", "Word count: time vs workers")

import pandas as pd
pd.DataFrame(records)

## 10 · Memoria dei worker

Da guardare fra una misura e l'altra durante i benchmark: se l'*unmanaged* cresce in
modo lineare con le ripetizioni c'è un hotspot di churn nel task; se oscilla attorno a
un plateau è solo working set (`docs/MEMORY_LEAK_REPORT.md`, §7.4).

`sweep` restituisce al sistema operativo la memoria che i worker trattengono senza più
usarla. Va chiamata **tra** le misure cronometrate, mai dentro una.

In [ ]:
for address, unmanaged in unmanaged_memory(client).items():
    print(f"{address:<28} unmanaged {unmanaged / 1e9:5.2f} GB")

sweep(client)

## 11 · Chiusura

In [ ]:
client.close()
if cluster is not None:
    cluster.close()
print("cluster chiuso")